In [15]:
import matplotlib.pyplot as plt
from abtem.parametrizations import LobatoParametrization
from ase.io import read

import abtem

abtem.config.set({"diagnostics.progress_bar": False});

(walkthrough:potentials)=
# Potentials
An electron beam interacts with a specimen principally through the Coulomb potential of its electrons and nuclei. Thus the total electrostatic potential of the sample is required for image simulations. Typically, a so-called independent atom model (IAM) is used, which calculates the potential as a superposition of atomic potentials, hence neglecting effects due to valence bonding.

## Atomic potential parametrization

The electron charge distribution of a single atom can be calculated from a first-principles electronic structure calculation, for example using the [Hartree-Fock method](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method) or [density functional theory](https://en.wikipedia.org/wiki/Density_functional_theory). Given a charge distribution, the potential can be obtained via [Poisson's equation](https://en.wikipedia.org/wiki/Poisson%27s_equation). Most multislice simulation codes include a parametrization of the atomic potentials, with a table of parameters for each element fitted to the potential calculated elsewhere from first principles. 

We show the radial dependence of the electrostatic potential and scattering  of five selected elements below. Note that the potentials tend to infinity at $r=0$ due to the point-like nuclear charge.

In [16]:
symbols = ["C", "N", "Si", "Au", "U"]

parametrization = LobatoParametrization()

potentials = parametrization.line_profiles(symbols, cutoff=2, name="potential")
scattering_factor = parametrization.line_profiles(
    symbols, cutoff=3, name="scattering_factor"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
visualization = potentials.show(ax=ax1, legend=False)
visualization.set_ylim([-1e2, 2e3])

scattering_factor.show(legend=True, ax=ax2);

The default parametrization in abTEM, which is shown above, is created by Ivan Lobato{cite}`lobato-parameters`. We also implement the parametrization by Earl J. Kirkland{cite}`kirkland` and the parametrization by Peng{cite}`peng-parameters`. The differences between the parametrizations are generally negligible at low scattering angles, but the parametrization by Lobato is more accurate at higher angles{cite}`lobato-parameters`.

# Independent atom model
The full specimen potential, $V(r)$, is usually obtained as a linear superposition of atomic potentials

$$
    V(r) = \sum_i V_i(r-r_i) \quad,
$$

where $V_i(r)$ is the atomic potential of the $i$'th atom. This is known as the independent atom model (IAM), and as a superposition of independent atoms, it neglects bonding effects. While this is in many cases an adequate approximation, there are situations where that is not the case ({cite}`madsen_ab_initio`) which *ab*TEM is particularly designed to address (as discussed [later in the walkthrough](walkthrough:charge_density).

## `Potential`
Here, we create a `Potential` object representing an IAM potential of SrTiO<sub>3</sub> with the Lobato parametrization. The `sampling` denotes the spacing of the $xy$-samples of the potential, `slice_thickness` determines the spacing the slices in direction of electron beam and `projection` determines how those slices are calculated (as described [later in this document](walkthrough:slicing)).

We also repeat the structure to make a larger supercell for visualizations.

In [17]:
srtio3 = read("./data/SrTiO3.cif")
repeated_srtio3 = srtio3 * (2, 2, 6)

potential = abtem.Potential(
    repeated_srtio3,
    sampling=0.05,
    parametrization="lobato",
    slice_thickness=1,
    projection="finite",
)

The potential has 24 slices along the $z$ propagation direction, as may be determined from getting its length.

In [18]:
len(potential)

24

The projected potential, i.e. the sum of all the slices multiplied by the thickness, may be shown using the `show` method.

In [19]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

abtem.show_atoms(repeated_srtio3, ax=ax1, legend=True)

visualization = (potential.build() * 0.1).show(ax=ax2);

The `Potential` may be indexed to return a subset of slices. Below we select the first five slices (using the Python syntax for indexing lists, `[:5]`) and show them all by setting `project=False` (the default is to show the projected potential) and `explode=True` (which automatically creates a multi-panel figure; in this instance the latter setting is assumed if the former is set). The titles of the panels are automatically set based on the depth of the slices, note that the slice thickness is not exactly $1 \ \mathrm{Å}$, it is rounded down to the closest value fitting an integer number of times.

In [20]:
visualization = potential[:5].show(
    project=False,
    explode=True,
    figsize=(14, 5),
    common_color_scale=True,
    cbar=True,
)

````{note} Interactive visualizations
You can use our interactive features to scroll through all the slices in your potential. 

To enable interactivity, you first have to enable a compatible matplotlib backend. We recommend using `ipympl`:

```{code}
%matplotlib ipympl
```

You can then create an interactive visualization by setting `interact=True` in the `.show` method. Make sure that `explode` is not set to `True`. 
````

## Building and saving the potential
The `Potential` does not store the calculated potential slices. Hence, if a simulation, such as STEM requires multiple propagations, each slice have to be calculated multiple times. For this reason, *ab*TEM often precalculates the potential whenever it has to be used more than once. 

The potential can be precalculated manually using the `build` method, but you should typically let *ab*TEM decide whether to precalculate the potential. Note that we also need to `compute` it, *ab*TEM by default uses so-called lazy arrays (see our [description of Dask](walkthrough:parallelization) for detail); a progress bar is displayed by default.

In [21]:
potential_array = potential.build().compute()

This returns an `PotentialArray` object, which stores each 2D potential slice in a 3D array. The first dimension is the slice index and the last two are the spatial dimensions.

In [22]:
potential_array.shape

(24, 158, 158)

The calculated potential can be stored in the open-source [Zarr file format](https://zarr.readthedocs.io/en/stable/) and conveniently read back in.

*New since v.1.0.10*: by specifying the file ending `.zip`, *abTEM* will automatically make a ZipStore and use Zstandard compression at default level 4.

In [23]:
potential_array.to_zarr("data/srtio3_repeated_potential.zarr", overwrite=True)

abtem.from_zarr("data/srtio3_repeated_potential.zarr");

(walkthrough:slicing)= 
## Slicing the potential

The multislice algorithm underlying most modern TEM image simulations requires a mathematical discretization of the potential into slices, which is exact in the limit of infinitely thin slices. The slice thickness can be considered a convergence parameter, and since more slices increases the computational cost, an optimum providing sufficient precision at an acceptable cost can be selected.

A reasonable value for slice thickness is generally between $0.5 \ \mathrm{Å}$ and $2 \ \mathrm{Å}$; our default value is $1 \ \mathrm{Å}$. *ab*TEM also provides multiple options for evaluating the integrals required for slicing the potential that make slightly different tradeoffs in terms of precision and performance.

### Finite projection integrals

*ab*TEM implements an accurate finite potential projection method. Numerical integration is used to calculate the integrals of the form

$$
V_{n}(x, y) = \int_{z_n}^{z_n+\Delta z} V(x,y,z) dz \quad ,
$$ (eq:potentials:slice)

where $z_n$ is the $z$-position at the entrance of the $n$'th slice and $\Delta z$ is the slice thickness.

We used the *ab*TEM default method of handling finite projection integrals above; additional options and a description of the methods is given in [an appendix](../appendix/potential_integrals.ipynb). 

### Infinite projection integrals

Since the finite projection method can be computationally demanding, *ab*TEM also implements potentials using infinite projection integrals. The finite integrals are replaced by infinite integrals, which may be evaluated analytically

$$ \int_{-\infty}^{\infty} V(x,y,z) dz \approx \int_{z_i}^{z_i+\Delta z} V(x,y,z) dz $$

The infinite projection of the atomic potential for each atom is assigned to a single slice. The implementation uses the hybrid real-space/Fourier-space approach by van den Broek {cite}`van_den_broek_infinite_projections`. 

Below we create the same SrTiO3 potential as above with infinite projections. The potential looks almost identical to the finite projection, but note that the slice at $z=2.96 \ \mathrm{Å}$ has zero intensity because there is not atoms in this slice.

In [24]:
potential_infinite = abtem.Potential(
    repeated_srtio3,
    sampling=0.05,
    parametrization="lobato",
    slice_thickness=1,
    projection="infinite",
)

visualization = potential_infinite[:5].show(
    project=False,
    figsize=(14, 5),
    common_color_scale=True,
    cbar=True,
);

Using infinite projections can be *much* faster, especially for potentials with a large numbers of atoms. As shown by the time below, the potential takes less than 0.5 s to calculate, which you may compare this to the finite projection calculated above that took 4 s. The error introduced by using infinite integrals is in most, but not all, cases negligible compared to other sources of error. 

In [25]:
potential_infinite.build().compute();

### Potentials depth profile

To see the differences between finite and infinite projection more clearly, we can plot the depth profiles of the potentials together with the slice markers using `.show_depth_profile()`.

Notice how especially for thinner slices, finite projection spreads the intensity of the atom across multiple slices, whereas infinite always keeps the entire atom within one slice. Even though the finite-projection side views appear dimmer, the projected potentials are essentially identical (as can be seen from the projected potentials plotted on the third row below).

In [26]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from ase import Atoms
import abtem

a = 3.905
cell = np.array([[a, 0.0, 0.0],
                 [0.0, a, 0.0],
                 [0.0, 0.0, a]])
scaled_positions = np.array([
    [0.0, 0.0, 0.0], [0.5, 0.5, 0.5], [0.5, 0.5, 0.0], [0.5, 0.0, 0.5], [0.0, 0.5, 0.5]
])
srtio3 = Atoms(symbols=['Sr', 'Ti', 'O', 'O', 'O'], scaled_positions=scaled_positions, cell=cell, pbc=True)
repeated_srtio3 = srtio3 * (2, 2, 6)

unit_z = srtio3.cell[2, 2]
vacuum = 3 * unit_z / 8
slicings = [('a/4', 4), ('a/8', 8), ('a/16', 16)]
GPTS = 256

repeated_srtio3.center(axis=2, vacuum=vacuum)
cell_z = repeated_srtio3.cell[2, 2]
cell_x = repeated_srtio3.cell[0, 0]
cell_y = repeated_srtio3.cell[1, 1]

z0 = sorted(set(np.round(repeated_srtio3.positions[:, 2], 6)))[0]

def slice_center_shift(st):
    # smallest-magnitude shift (positive or negative) that centers the atom
    # in its assigned slice -- avoids the near-full-slice-thickness artifact
    # that a plain non-negative modulo produces when the atom is already
    # (near-)centered.
    return (st / 2 - z0 + st / 2) % st - st / 2

fig = plt.figure(figsize=(8, 14))
gs = gridspec.GridSpec(3, 4, figure=fig,
                       height_ratios=[cell_z, cell_z, cell_x],
                       hspace=0.1, wspace=0.05)

axes = np.empty((3, 4), dtype=object)
for col in range(4):
    axes[0, col] = fig.add_subplot(gs[0, col])
for col in range(4):
    axes[1, col] = fig.add_subplot(gs[1, col], sharex=axes[0, col], sharey=axes[0, col])
for col in range(4):
    sharey = axes[2, 0] if col > 0 else None
    axes[2, col] = fig.add_subplot(gs[2, col], sharex=axes[0, col], sharey=sharey)

for row in range(2):
    ax_at = axes[row, 0]
    abtem.show_atoms(repeated_srtio3, plane='xz', show_periodic=True, scale=0.5, ax=ax_at,
                     tight_limits=True, legend=False)
    ax_at.set_title('Side view')

for col, (label, st_frac) in enumerate(slicings):
    st = unit_z / st_frac
    delta = slice_center_shift(st)

    atoms_col = repeated_srtio3.copy()
    atoms_col.positions[:, 2] += delta

    pot_inf = abtem.Potential(atoms_col.copy(), gpts=(GPTS, GPTS), slice_thickness=st,
                             projection='infinite')
    pot_fin = abtem.Potential(atoms_col.copy(), gpts=(GPTS, GPTS), slice_thickness=st,
                             projection='finite')

    pa_inf = pot_inf.build().compute()
    pa_fin = pot_fin.build().compute()
    dp_inf = pa_inf.depth_profile()
    dp_fin = pa_fin.depth_profile()
    vmax = max(float(dp_inf.array.max()), float(dp_fin.array.max()))

    pot_inf.show_depth_profile(ax=axes[0, col+1], vmin=0, vmax=vmax)
    axes[0, col+1].set_title(f'Inf. (dz={st:.2f} Å)')

    pot_fin.show_depth_profile(ax=axes[1, col+1], vmin=0, vmax=vmax)
    axes[1, col+1].set_title(f'Fin. (dz={st:.2f} Å)')

st4 = unit_z / 4
delta4 = slice_center_shift(st4)
atoms_proj = repeated_srtio3.copy()
atoms_proj.positions[:, 2] += delta4

proj_inf = abtem.Potential(atoms_proj.copy(), gpts=(GPTS, GPTS), slice_thickness=st4,
                           projection='infinite').build().compute().project()
proj_fin = abtem.Potential(atoms_proj.copy(), gpts=(GPTS, GPTS), slice_thickness=st4,
                           projection='finite').build().compute().project()
proj_vmax = max(float(proj_inf.array.max()), float(proj_fin.array.max()))

abtem.show_atoms(repeated_srtio3, show_periodic=True, scale=0.5, ax=axes[2, 0], legend=False,
                 tight_limits=True)
axes[2, 0].set_title('Top view')

proj_inf.show(ax=axes[2, 1], vmin=0, vmax=proj_vmax)
axes[2, 1].set_title('Inf.')

proj_fin.show(ax=axes[2, 2], vmin=0, vmax=proj_vmax)
axes[2, 2].set_title('Fin.')

diff = proj_inf - proj_fin
diff.show(cmap='RdBu_r', ax=axes[2, 3], vmin=-proj_vmax, vmax=proj_vmax)
axes[2, 3].set_title('Inf. – Fin.')

for row in range(2):
    for col in range(4):
        axes[row, col].set_xlabel('')
        plt.setp(axes[row, col].get_xticklabels(), visible=False)

for row in range(3):
    for col in range(1, 4):
        axes[row, col].set_ylabel('')
        plt.setp(axes[row, col].get_yticklabels(), visible=False)

for col in range(4):
    axes[2, col].set_xlabel(r'$x$ [Å]')

axes[0, 0].set_ylabel(r'$z$ [Å]')
axes[1, 0].set_ylabel(r'$z$ [Å]')
axes[2, 0].set_ylabel(r'$y$ [Å]');

## Crystal potentials

Calculating the potential is generally not a significant cost for simulations where the same potential is used in many runs of the multislice algorithm. However, for simulations that require just one or a few wave functions, such as HRTEM and CBED, it can also be an advantage to use the a periodic crystal potential.

The `CrystalPotential` allows fast calculation for potentials of crystals by tiling a repeating unit of the potential. Below we create a `CrystalPotential` by tiling the already calculated finite projected potential.

In [27]:
crystal_potential = abtem.CrystalPotential(
    potential_unit=potential, repetitions=(5, 5, 10), seeds=None
)

crystal_potential.build().array

dask.array<_wrap_build_potential, shape=(240, 790, 790), dtype=float32, chunksize=(240, 790, 790), chunktype=numpy.ndarray>

In [28]:
crystal_potential.build().compute();

We also implement a fast frozen phonon algorithm for crystal potentials by reshuffling precalculated slices {cite}`drprobe`; see our walkthrough on [frozen phonons](walkthrough:frozen_phonons).